# Nyando Flood Monitor - Sprint 3 Fine-Tuning
Run this notebook on Google Colab to utilize free GPUs. Ensure you have uploaded the `Nyando_Dataset` folder and the `unet_sen1floods_best.pth` base weights to your Google Drive.

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install rasterio

In [ ]:
# Extract the dataset zip on Drive (skip if already unzipped)
import zipfile, os
ZIP_PATH = '/content/drive/MyDrive/Nyando_Dataset.zip'
DATASET_DIR = '/content/drive/MyDrive/Nyando_Dataset'
if os.path.exists(ZIP_PATH) and not os.path.exists(os.path.join(DATASET_DIR, 'LabelHand')):
    os.makedirs(DATASET_DIR, exist_ok=True)
    print('Extracting', ZIP_PATH, '->', DATASET_DIR)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATASET_DIR)
    print('Done.')
else:
    print('Dataset already extracted at', DATASET_DIR)


In [ ]:
# 2. Imports and Dataset Setup
import os
import csv
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import rasterio
import numpy as np
import random

class NyandoFloodDataset(Dataset):
    def __init__(self, dataset_dir, split="train", transform=True):
        self.dataset_dir = dataset_dir
        self.transform = transform
        self.patches = []

        csv_path = os.path.join(dataset_dir, f"nyando_split_{split}.csv")
        with open(csv_path, 'r') as f:
            reader = csv.reader(f)
            for row in reader:
                if row:
                    self.patches.append(row[0])

        # Oversample flood-positive patches (flood frac > 0.5%) so the many
        # near-empty patches don't dominate. Flood-free patches (mostly the
        # hand-labelled ones) stay at weight 1.0 -> hard negatives.
        self.sample_weights = None
        if split == "train":
            self.sample_weights = []
            for pid in self.patches:
                with rasterio.open(os.path.join(
                        dataset_dir, "LabelHand", f"{pid}_LabelHand.tif")) as src:
                    m = src.read(1)
                frac = float((m == 1).mean())
                self.sample_weights.append(3.0 if frac > 0.005 else 1.0)

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        patch_id = self.patches[idx]

        sar_path = os.path.join(self.dataset_dir, "S1Hand", f"{patch_id}_S1Hand.tif")
        mask_path = os.path.join(self.dataset_dir, "LabelHand", f"{patch_id}_LabelHand.tif")

        with rasterio.open(sar_path) as src:
            sar = src.read().astype(np.float32)

        # Fixed dB scaling — preserves absolute backscatter (open water is
        # very low dB), unlike per-patch z-score which erases it.
        # Must match pipeline/05_inference.py normalize_patch.
        sar = 10.0 * np.log10(np.where(sar > 0, sar, np.nan))
        sar = np.nan_to_num(sar, nan=-15.0)
        sar = np.clip(sar, -30.0, 0.0)
        sar = (sar + 30.0) / 30.0   # -> [0, 1]

        with rasterio.open(mask_path) as src:
            mask = src.read(1).astype(np.float32)

        if self.transform:
            if random.random() > 0.5:
                sar = np.flip(sar, axis=1)
                mask = np.flip(mask, axis=0)
            if random.random() > 0.5:
                sar = np.flip(sar, axis=2)
                mask = np.flip(mask, axis=1)
            k = random.randint(0, 3)
            sar = np.rot90(sar, k, axes=(1, 2))
            mask = np.rot90(mask, k, axes=(0, 1))

        sar = np.ascontiguousarray(sar)
        mask = np.ascontiguousarray(mask)

        return torch.from_numpy(sar), torch.from_numpy(mask).unsqueeze(0)

In [4]:
# 3. Model Architecture
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(256, 512)
        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(128, 64)
        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))
        x_btn = self.bottleneck(self.pool3(x3))
        x = self.conv1(torch.cat([self.up1(x_btn), x3], dim=1))
        x = self.conv2(torch.cat([self.up2(x), x2], dim=1))
        x = self.conv3(torch.cat([self.up3(x), x1], dim=1))
        return self.sigmoid(self.out_conv(x))

In [5]:
# 4. Loss Function and Metrics
class MaskedBCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss(reduction='none')

    def forward(self, pred, target):
        valid_mask = (target != -1).float()
        safe_target = target.clamp(min=0)
        bce_loss = self.bce(pred, safe_target)
        bce_loss = (bce_loss * valid_mask).sum() / (valid_mask.sum() + 1e-8)
        smooth = 1e-5
        pred_flat = pred.view(-1)
        target_flat = safe_target.view(-1)
        mask_flat = valid_mask.view(-1)
        intersection = (pred_flat * target_flat * mask_flat).sum()
        denominator = (pred_flat * mask_flat).sum() + (target_flat * mask_flat).sum()
        dice_loss = 1 - ((2. * intersection + smooth) / (denominator + smooth))
        return bce_loss + dice_loss

def calculate_metrics(pred, target):
    valid_mask = (target != -1)
    preds_bin = (pred > 0.5).bool() & valid_mask
    targets_bin = (target > 0).bool() & valid_mask
    tp = (preds_bin & targets_bin).float().sum()
    fp = (preds_bin & ~targets_bin & valid_mask).float().sum()
    fn = (~preds_bin & targets_bin & valid_mask).float().sum()
    precision = (tp / (tp + fp + 1e-8)).item()
    recall = (tp / (tp + fn + 1e-8)).item()
    f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).item()
    iou = (tp / (tp + fp + fn + 1e-8)).item()
    return iou, f1, precision, recall

In [ ]:
# 5. Training Loop — pooled val IoU checkpointing + LR scheduler
def train_model(dataset_dir, base_weights_path="unet_sen1floods_best.pth",
                epochs=80, batch_size=8, lr=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_dataset = NyandoFloodDataset(dataset_dir, split="train", transform=True)
    val_dataset = NyandoFloodDataset(dataset_dir, split="val", transform=False)

    if train_dataset.sample_weights is not None:
        sampler = WeightedRandomSampler(
            train_dataset.sample_weights,
            num_samples=len(train_dataset.sample_weights),
            replacement=True)
        train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                  sampler=sampler)
        print(f"Oversampling enabled: {sum(1 for w in train_dataset.sample_weights if w > 1)} "
              f"flood-positive patches upweighted")
    else:
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = UNet(in_channels=2, out_channels=1).to(device)

    if os.path.exists(base_weights_path):
        model.load_state_dict(torch.load(base_weights_path, map_location=device))
        print(f"Successfully loaded base weights from {base_weights_path}")
    else:
        print(f"WARNING: Base weights '{base_weights_path}' not found!")

    criterion = MaskedBCEDiceLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5)

    patience = 12
    best_iou = 0.0
    epochs_no_improve = 0
    save_path = "/content/drive/MyDrive/nyando_finetuned_model.pth"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    history = {"train_loss": [], "val_loss": [], "val_iou": [], "val_f1": []}

    print(f"Starting Fine-Tuning for {epochs} epochs...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for sar, mask in train_loader:
            sar, mask = sar.to(device), mask.to(device)
            optimizer.zero_grad()
            outputs = model(sar)
            loss = criterion(outputs, mask)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        # Pooled TP/FP/FN over the whole val split — checkpoint selection
        # and scheduler both use this (excludes -1 ignore pixels).
        tp = fp = fn = 0
        with torch.no_grad():
            for sar, mask in val_loader:
                sar, mask = sar.to(device), mask.to(device)
                outputs = model(sar)
                val_loss += criterion(outputs, mask).item()
                valid = (mask != -1)
                p = (outputs > 0.5).bool() & valid
                t = (mask > 0).bool() & valid
                tp += (p & t).sum().item()
                fp += (p & ~t).sum().item()
                fn += (~p & t).sum().item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        avg_val_iou = tp / (tp + fp + fn + 1e-8)
        avg_val_f1 = 2 * tp / (2 * tp + fp + fn + 1e-8)
        scheduler.step(avg_val_iou)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_iou"].append(avg_val_iou)
        history["val_f1"].append(avg_val_f1)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val IoU: {avg_val_iou:.4f} | "
              f"Val F1: {avg_val_f1:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        if avg_val_iou > best_iou:
            best_iou = avg_val_iou
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
            print(f" -> Best model saved with IoU: {best_iou:.4f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break

    return history

In [ ]:
# Dataset overview — patches per split/date and label flood fraction
DATASET_DIR = '/content/drive/MyDrive/Nyando_Dataset'
import matplotlib.pyplot as plt
import numpy as np, os, csv, re
from collections import Counter

_DATED = re.compile(r'^Nyando_(\d{4}-\d{2}-\d{2})_\d+$')

for split in ['train', 'val', 'test']:
    ids = [r[0] for r in csv.reader(open(f'{DATASET_DIR}/nyando_split_{split}.csv')) if r]
    by_date = Counter(
        (_DATED.match(i).group(1) if _DATED.match(i) else 'hand-labelled') for i in ids
    )
    print(f'{split}: {len(ids)} patches -> {dict(by_date)}')

# Flood fraction histogram over label patches (sample up to 300)
fracs = []
label_dir = os.path.join(DATASET_DIR, 'LabelHand')
for f in sorted(os.listdir(label_dir)):
    if f.endswith('.tif') and len(fracs) < 300:
        with rasterio.open(os.path.join(label_dir, f)) as src:
            fracs.append(float((src.read(1) == 1).mean()))

plt.figure(figsize=(8, 4))
plt.hist(fracs, bins=20, color='steelblue', edgecolor='black')
plt.xlabel('Flood fraction of patch')
plt.ylabel('Number of patches')
plt.title(f'Label flood fraction (n={len(fracs)}, mean={np.mean(fracs):.3f})')
plt.show()

# Flag suspicious labels: patches that are >95% flood or exactly 0%
all_fracs = []
for f in sorted(os.listdir(label_dir)):
    if f.endswith('.tif'):
        with rasterio.open(os.path.join(label_dir, f)) as src:
            all_fracs.append((f, float((src.read(1) == 1).mean())))
suspicious = [f for f, v in all_fracs if v > 0.95 or v == 0.0]
print(f'{len(suspicious)}/{len(all_fracs)} patches are degenerate (>95% or 0% flood)')
if suspicious[:10]:
    print('examples:', suspicious[:10])

In [ ]:
DRIVE_DIR   = '/content/drive/MyDrive'
DATASET_DIR = DRIVE_DIR + '/Nyando_Dataset'
BASE_WEIGHTS = DRIVE_DIR + '/unet_sen1floods_best.pth'

history = train_model(DATASET_DIR, base_weights_path=BASE_WEIGHTS)

In [ ]:
# Training curves — loss, IoU, F1 per epoch
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(epochs, history['val_iou'], color='seagreen')
axes[1].set_title('Validation IoU'); axes[1].set_xlabel('epoch')

axes[2].plot(epochs, history['val_f1'], color='darkorange')
axes[2].set_title('Validation F1'); axes[2].set_xlabel('epoch')

plt.tight_layout()
plt.show()

best_epoch = int(np.argmax(history['val_iou'])) + 1
print(f"Best val IoU {max(history['val_iou']):.4f} at epoch {best_epoch}")

In [ ]:
# Visualise predictions — SAR VV, ground truth, prediction, overlay
import matplotlib.pyplot as plt
import numpy as np, torch, os

DATASET_DIR = '/content/drive/MyDrive/Nyando_Dataset'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = UNet(in_channels=2, out_channels=1).to(device)
model.load_state_dict(torch.load(os.path.join(DATASET_DIR, 'nyando_finetuned_model.pth'), map_location=device))
model.eval()

val_ds = NyandoFloodDataset(DATASET_DIR, split='val', transform=False)
n_show = min(6, len(val_ds))
# spread picks across the val set deterministically
pick = np.linspace(0, len(val_ds) - 1, n_show).astype(int)

fig, axes = plt.subplots(n_show, 4, figsize=(14, 3.2 * n_show))
if n_show == 1:
    axes = axes[None, :]

for row, idx in enumerate(pick):
    sar, mask = val_ds[idx]
    with torch.no_grad():
        pred = model(sar.unsqueeze(0).to(device)).squeeze().cpu().numpy()

    vv_raw = sar[0].numpy()                   # VV channel (z-scored)
    vv = (vv_raw - vv_raw.min()) / (vv_raw.max() - vv_raw.min() + 1e-8)  # -> [0,1]
    gt = mask.squeeze().numpy()               # 0/1
    prob = pred                                # 0..1 probability
    pb = (pred > 0.5).astype(float)

    # VV with prediction overlay (red channel boosted where flood predicted)
    overlay = np.stack([vv, vv, vv], axis=-1)
    overlay[..., 0] = np.clip(vv + pb * 0.6, 0, 1)

    titles = ['SAR VV', 'Ground truth', 'Prediction prob', 'VV + pred overlay']
    for col, (img, title, kw) in enumerate(zip(
            [vv, gt, prob, overlay], titles,
            [dict(cmap='gray'), dict(cmap='Reds', vmin=0, vmax=1),
             dict(cmap='hot', vmin=0, vmax=1), None])):
        ax = axes[row, col]
        if kw is None:
            ax.imshow(img)
        else:
            ax.imshow(img, **kw)
        if row == 0:
            ax.set_title(title)
        ax.axis('off')

    iou, f1, prec, rec = calculate_metrics(
        torch.from_numpy(pred).reshape(1, 1, *pred.shape),
        mask.reshape(1, 1, *mask.shape[1:]))
    axes[row, 0].text(0.02, 0.95, f'patch {idx}\nIoU={iou:.3f} F1={f1:.3f}',
                      transform=axes[row, 0].transAxes, color='yellow',
                      fontsize=8, va='top',
                      bbox=dict(facecolor='black', alpha=0.5, pad=2))

plt.tight_layout()
plt.show()

In [8]:
# Test Set Evaluation
import torch
from torch.utils.data import DataLoader

DATASET_DIR = '/content/drive/MyDrive/Nyando_Dataset'
save_path   = '/content/drive/MyDrive/Nyando_Dataset/nyando_finetuned_model.pth'
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = UNet(in_channels=2, out_channels=1).to(device)
model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

test_dataset = NyandoFloodDataset(DATASET_DIR, split="test", transform=False)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)

test_iou = test_f1 = test_prec = test_rec = 0

with torch.no_grad():
    for sar, mask in test_loader:
        sar, mask = sar.to(device), mask.to(device)
        iou, f1, prec, rec = calculate_metrics(model(sar), mask)
        test_iou += iou; test_f1 += f1; test_prec += prec; test_rec += rec

n = len(test_loader)
print(f"Test IoU:       {test_iou/n:.4f}")
print(f"Test F1:        {test_f1/n:.4f}")
print(f"Test Precision: {test_prec/n:.4f}")
print(f"Test Recall:    {test_rec/n:.4f}")


Test IoU:       0.9624
Test F1:        0.9805
Test Precision: 0.9944
Test Recall:    0.9678
